In [23]:
from db_connector import list_all_tables, load_table_to_df
import pandas as pd
import numpy as np
import datetime
from lightfm import LightFM
from lightfm.data import Dataset
from lightfm.evaluation import precision_at_k, recall_at_k

# 1. 데이터 로드
all_tables = list_all_tables()
all_dfs = {}
for table in all_tables:
    try:
        all_dfs[table] = load_table_to_df(table)
    except:
        pass
print("데이터 로드 완료!")

데이터 로드 완료!


In [24]:
# 1. LightFM 매트릭스 구성 함수
def build_lightfm_matrices(all_dfs):
    profiles = all_dfs['user_profiles'].copy()
    foods = all_dfs['foods'].copy()
    interactions = all_dfs.get('user_item_interactions', pd.DataFrame()).copy()
    
    # User Feature 가공: age_years_snapshot -> age_group (10단위)
    profiles['age_group'] = (profiles['age_years_snapshot'] // 10 * 10).astype(int)
    
    # Dataset 정의 (Item은 Food 단위)
    dataset = Dataset()
    
    # Feature 목록 정의
    user_feat_cols = ['sex', 'goal_type', 'age_group', 'activity_factor']
    item_feat_cols = ['category', 'budget_level', 'is_high_protein', 'is_low_fat']
    # meal_type은 interaction이나 candidate_items에서 와야 함 (여기서는 foods에 없으므로 생략하거나 다른 방식으로 처리)
    
    user_features_iter = []
    for col in user_feat_cols:
        for val in profiles[col].unique():
            user_features_iter.append(f"{col}:{val}")
            
    item_features_iter = []
    for col in item_feat_cols:
        for val in foods[col].unique():
            item_features_iter.append(f"{col}:{val}")

    dataset.fit(
        users=profiles['user_id'].unique(),
        items=foods['food_id'].unique(),
        user_features=user_features_iter,
        item_features=item_features_iter
    )

    # User Features Build
    user_feat_data = []
    for _, row in profiles.iterrows():
        feats = [f"{col}:{row[col]}" for col in user_feat_cols]
        user_feat_data.append((row['user_id'], feats))
    user_features = dataset.build_user_features(user_feat_data)

    # Item Features Build
    item_feat_data = []
    for _, row in foods.iterrows():
        feats = [f"{col}:{row[col]}" for col in item_feat_cols]
        item_feat_data.append((row['food_id'], feats))
    item_features = dataset.build_item_features(item_feat_data)

    # Interaction Matrix Build (1:9 분리 포함)
    if not interactions.empty and 'food_id' in interactions.columns:
        # valid한 food_id만 필터링
        interactions = interactions[interactions['food_id'].isin(foods['food_id'])]
        
        # 1:9 분리 (Shuffle 후 분할)
        interactions = interactions.sample(frac=1, random_state=42).reset_index(drop=True)
        split_idx = int(len(interactions) * 0.1)
        train_df = interactions.iloc[:split_idx]
        test_df = interactions.iloc[split_idx:]
        
        (train_interactions, train_weights) = dataset.build_interactions(
            [(x['user_id'], x['food_id'], x['interaction_weight']) for _, x in train_df.iterrows()]
        )
        (test_interactions, test_weights) = dataset.build_interactions(
            [(x['user_id'], x['food_id'], x['interaction_weight']) for _, x in test_df.iterrows()]
        )
    else:
        train_interactions, train_weights = None, None
        test_interactions, test_weights = None, None
        
    return dataset, user_features, item_features, train_interactions, train_weights, test_interactions

dataset, user_features, item_features, train_interactions, train_weights, test_interactions = build_lightfm_matrices(all_dfs)
print("매트릭스 빌드 완료!")

매트릭스 빌드 완료!


In [29]:
# 2. LightFM 매트릭스 확인 함수
def inspect_lightfm_matrices(dataset, user_features, item_features, interaction_matrix):
    n_users, n_items = dataset.interactions_shape()
    user_id_map, user_feature_map, item_id_map, item_feature_map = dataset.mapping()
    
    # 1. 매트릭스 차원(Shape) 정보
    print("="*50)
    print("[1. 매트릭스 구조 요약]")
    print(f" - User Feature Matrix: {user_features.shape} (유저 수 x [유저ID + 피처태그 수])")
    print(f" - Item Feature Matrix: {item_features.shape} (식품 수 x [식품ID + 피처태그 수])")
    if interaction_matrix is not None:
        print(f" - Interaction Matrix: {interaction_matrix.shape} (유저 수 x 식품 수)")
    print("="*50)

    # 2. 열(Column) 구성 상세
    print("\n[2. 매트릭스 열(Column) 구성 상세]")
    print(f" - 유저 매트릭스 열 구성 (총 {len(user_feature_map)}개):")
    print(f"   * 앞부분 ({n_users}개): 유저 고유 ID (Identity Features)")
    print(f"   * 뒷부분 ({len(user_feature_map) - n_users}개): 성별, 목표, 연령대 등 속성 (Metadata Features)")
    
    print(f"\n - 식품 매트릭스 열 구성 (총 {len(item_feature_map)}개):")
    print(f"   * 앞부분 ({n_items}개): 식품 고유 ID (Identity Features)")
    print(f"   * 뒷부분 ({len(item_feature_map) - n_items}개): 카테고리, 예산레벨 등 속성 (Metadata Features)")

    # 3. 특정 유저를 통한 매트릭스 매핑 예시
    print("\n" + "="*50)
    print("[3. 데이터 매핑 예시 (User ID 샘플)]")
    sample_user_id = list(user_id_map.keys())[0] if user_id_map else None
    if sample_user_id is not None:
        u_idx = user_id_map[sample_user_id]
        u_row = user_features.getrow(u_idx)
        # 역맵핑으로 피처 이름 찾기
        inv_user_feat_map = {v: k for k, v in user_feature_map.items()}
        active_features = [inv_user_feat_map[i] for i in u_row.indices]
        
        print(f" - 유저 ID {sample_user_id}의 내부 인덱스: {u_idx}")
        print(f" - 이 유저의 행에서 '1'로 표시된 열(Active Features):")
        for feat in active_features:
            print(f"   > {feat}")
    print("="*50)

    # 4. 희소성(Sparsity) 정보
    if interaction_matrix is not None:
        sparsity = (1.0 - (interaction_matrix.nnz / float(n_users * n_items))) * 100
        print(f"\n[4. 상호작용 정보]")
        print(f" - Non-zero Interaction count: {interaction_matrix.nnz}")
        print(f" - Sparsity (비어있는 비율): {sparsity:.2f}%")
    else:
        print("\n[4. 상호작용 정보] No interaction data available.")

inspect_lightfm_matrices(dataset, user_features, item_features, train_interactions)

[1. 매트릭스 구조 요약]
 - User Feature Matrix: (10, 18) (유저 수 x [유저ID + 피처태그 수])
 - Item Feature Matrix: (16134, 16157) (식품 수 x [식품ID + 피처태그 수])
 - Interaction Matrix: (10, 16134) (유저 수 x 식품 수)

[2. 매트릭스 열(Column) 구성 상세]
 - 유저 매트릭스 열 구성 (총 18개):
   * 앞부분 (10개): 유저 고유 ID (Identity Features)
   * 뒷부분 (8개): 성별, 목표, 연령대 등 속성 (Metadata Features)

 - 식품 매트릭스 열 구성 (총 16157개):
   * 앞부분 (16134개): 식품 고유 ID (Identity Features)
   * 뒷부분 (23개): 카테고리, 예산레벨 등 속성 (Metadata Features)

[3. 데이터 매핑 예시 (User ID 샘플)]
 - 유저 ID 1의 내부 인덱스: 0
 - 이 유저의 행에서 '1'로 표시된 열(Active Features):
   > 1
   > sex:male
   > goal_type:cut
   > age_group:20
   > activity_factor:1.55

[4. 상호작용 정보]
 - Non-zero Interaction count: 0
 - Sparsity (비어있는 비율): 100.00%


In [26]:
# 3. LightFM 학습 함수
def train_lightfm_model(interaction_matrix, user_features, item_features, sample_weight=None):
    model = LightFM(loss='warp', no_components=30, learning_rate=0.05, random_state=42)
    
    if interaction_matrix is not None:
        model.fit(
            interaction_matrix, 
            user_features=user_features, 
            item_features=item_features, 
            sample_weight=sample_weight, 
            epochs=20, 
            verbose=True
        )
    else:
        # 데이터가 없을 경우 피처 기반으로만 동작하도록 더미 피크 설정 (실제 학습은 안됨)
        print("No interactions for training. Model will use initial feature embeddings.")
        
    return model

model = train_lightfm_model(train_interactions, user_features, item_features, train_weights)

Epoch: 100%|██████████| 20/20 [00:00<00:00, 1273.74it/s]


In [27]:
# 4. LightFM 예측 함수 (식단 점수 도출)
def predict_meal_scores(user_id, meal_candidates, all_dfs, model, dataset, user_features, item_features):
    cand_items = all_dfs['meal_candidate_items']
    foods = all_dfs['foods'].set_index('food_id')
    
    user_id_map, _, item_id_map, _ = dataset.mapping()
    if user_id not in user_id_map:
        return {cand['candidate_id']: 0.0 for cand in meal_candidates}
    
    user_idx = user_id_map[user_id]
    meal_scores = {}
    
    for cand in meal_candidates:
        cid = cand['candidate_id']
        items = cand_items[cand_items['candidate_id'] == cid]
        
        if items.empty:
            meal_scores[cid] = 0.0
            continue
            
        total_weighted_score = 0.0
        total_serving_size = 0.0
        
        for _, item in items.iterrows():
            fid = item['food_id']
            if fid not in item_id_map:
                continue
                
            # 개별 식품 점수 예측
            food_idx = item_id_map[fid]
            food_score = model.predict(user_idx, np.array([food_idx]), user_features=user_features, item_features=item_features)[0]
            
            # serving_size_g 비중 계산 (데이터에 없을 경우 1.0 기본값)
            serving_size = foods.loc[fid, 'serving_size_g'] if fid in foods.index else 1.0
            if pd.isna(serving_size) or serving_size <= 0: serving_size = 1.0
            
            total_weighted_score += food_score * serving_size
            total_serving_size += serving_size
            
        meal_scores[cid] = total_weighted_score / total_serving_size if total_serving_size > 0 else 0.0
        
    return meal_scores

# 예측 테스트
if 'meal_candidates' in all_dfs:
    test_cands = all_dfs['meal_candidates'].head(5).to_dict('records')
    scores = predict_meal_scores(1, test_cands, all_dfs, model, dataset, user_features, item_features)
    print("\n[Meal Prediction Scores]")
    for cid, score in scores.items():
        print(f"Candidate ID {cid}: {score:.4f}")


[Meal Prediction Scores]
Candidate ID 50000: 0.0001
Candidate ID 50001: -0.0001
Candidate ID 50002: -0.0001
Candidate ID 50003: 0.0000
Candidate ID 50004: -0.0000
